# Part 1：來源 Ground-truth Runtime 的跨裝置 Scaling

## 研究問題

已知來源 GPU 的真實 iteration runtime 時，能否預測另一張 GPU 的 runtime？

## 目前結論

最佳 combined log_source_ols：MAPE 4.21%、R² 0.9843。這是 feasibility baseline，不是 production solution，因為部署時仍需先取得來源真實 runtime。

> 本 Notebook 新增於既有研究產物之外，不會取代原始 Python scripts 或 CSV／Markdown 報告。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 實驗步驟

1. 讀取兩張 GPU 的 baseline ground truth。
2. 依 workload 建立雙向 source→target 樣本。
3. 使用 direction-specific leave-one-workload-out。
4. 比較固定比例、線性與 log-source 模型。
5. 彙整 OOF error 與方向別結果。

## 輸入與 provenance

下列 cell 只檢查輸入是否存在，不修改資料。

In [ ]:
INPUTS = [
    "analysis_outputs/ground_truth.csv",
    "analysis_outputs/cross_device_samples.csv",
]
display(inspect_inputs(INPUTS))

## 重現入口

下列命令依現有 pipeline 排列。預設只列印；將 RUN_EXPENSIVE 明確改成 True 才會執行。

In [ ]:
COMMANDS = [
    ["analysis/run_baseline_models.py"],
]
run_commands(COMMANDS)

## 結果表

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part1_baselines/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part1_baselines/predictions.csv"

metrics = load_csv(METRICS_PATH)
sort_column = "mape_percent" if "mape_percent" in metrics.columns else "mean_period_ape_percent"
display(metrics.sort_values(sort_column).reset_index(drop=True))

if "direction" in metrics.columns and "model" in metrics.columns:
    plot_data = metrics.sort_values(sort_column).groupby("direction", as_index=False).first()
    ax = plot_data.plot.bar(x="direction", y=sort_column, legend=False, figsize=(8, 4))
    ax.set_ylabel(sort_column)
    ax.set_title("各方向最佳列（依表內指標排序）")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## OOF 預測與 worst cases

顯示現有 prediction table 中 APE 最大的列，並提供 actual-vs-predicted 圖。

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part1_baselines/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part1_baselines/predictions.csv"

predictions = load_csv(PREDICTIONS_PATH)
display(predictions.sort_values("ape_percent", ascending=False).head(15).reset_index(drop=True))

required = {"actual_ms", "predicted_ms", "ape_percent"}
if required.issubset(predictions.columns):
    view = predictions.copy()
    if "deployment_valid" in view.columns:
        view = view[view["deployment_valid"].astype(str).str.lower().eq("true")]
    ax = view.plot.scatter(x="actual_ms", y="predicted_ms", c="ape_percent", cmap="viridis", figsize=(6, 5))
    low = min(view["actual_ms"].min(), view["predicted_ms"].min())
    high = max(view["actual_ms"].max(), view["predicted_ms"].max())
    ax.plot([low, high], [low, high], "--", color="black", linewidth=1)
    ax.set_title("OOF prediction：actual vs predicted")
    plt.tight_layout()
    plt.show()

## 既有最終報告快照

> 來源：`analysis_outputs/model_experiments/part1_baselines/report.md`。這是建立 Notebook 時嵌入的結果快照；重新執行分析後，請以磁碟上的最新報告為準。

# Part 1: leakage-safe runtime baselines

Each direction is trained independently. Every outer fold holds out one workload, and the reverse direction is never used as training data for that fold.

| Direction | Model | MAE ms | RMSE ms | MAPE | Median APE | Max APE | R2 |
|---|---|---:|---:|---:|---:|---:|---:|
| RTX4090_to_RTX5090 | log_source_ols | 1.835 | 2.342 | 4.10% | 3.67% | 11.97% | 0.9852 |
| RTX4090_to_RTX5090 | global_ratio_median | 2.260 | 3.802 | 4.40% | 3.66% | 14.29% | 0.9609 |
| RTX4090_to_RTX5090 | linear_source_ols | 1.999 | 2.658 | 4.42% | 4.10% | 11.25% | 0.9809 |
| RTX4090_to_RTX5090 | identity_source | 5.854 | 8.068 | 11.93% | 10.43% | 25.93% | 0.8241 |
| RTX4090_to_RTX5090 | target_train_median | 15.568 | 22.278 | 30.33% | 25.95% | 67.05% | -0.3414 |
| RTX5090_to_RTX4090 | global_ratio_median | 2.493 | 4.191 | 4.23% | 3.80% | 12.50% | 0.9702 |
| RTX5090_to_RTX4090 | log_source_ols | 2.262 | 3.137 | 4.32% | 3.58% | 11.45% | 0.9833 |
| RTX5090_to_RTX4090 | linear_source_ols | 2.494 | 3.355 | 4.83% | 4.24% | 11.28% | 0.9809 |
| RTX5090_to_RTX4090 | identity_source | 5.854 | 8.068 | 10.40% | 9.45% | 20.59% | 0.8894 |
| RTX5090_to_RTX4090 | target_train_median | 18.821 | 27.601 | 32.67% | 30.13% | 70.38% | -0.2943 |
| both_directions_macro_rows | log_source_ols | 2.048 | 2.768 | 4.21% | 3.58% | 11.97% | 0.9843 |
| both_directions_macro_rows | global_ratio_median | 2.377 | 4.001 | 4.31% | 3.72% | 14.29% | 0.9672 |
| both_directions_macro_rows | linear_source_ols | 2.246 | 3.027 | 4.63% | 4.23% | 11.28% | 0.9812 |
| both_directions_macro_rows | identity_source | 5.854 | 8.068 | 11.16% | 9.88% | 25.93% | 0.8666 |
| both_directions_macro_rows | target_train_median | 17.194 | 25.082 | 31.50% | 28.22% | 70.38% | -0.2894 |

## Worst workloads for the best model in each direction

### RTX4090_to_RTX5090: `log_source_ols`

| Workload | Source ms | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|---:|
| C10 | 49.200 | 39.302 | 44.008 | 11.97% |
| C17 | 38.002 | 31.537 | 34.671 | 9.94% |
| C15 | 38.872 | 33.221 | 35.338 | 6.37% |
| C12 | 131.249 | 104.224 | 110.715 | 6.23% |
| C21 | 35.460 | 34.129 | 32.254 | 5.50% |

### RTX5090_to_RTX4090: `global_ratio_median`

| Workload | Source ms | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|---:|
| C12 | 104.224 | 131.249 | 114.837 | 12.50% |
| C10 | 39.302 | 49.200 | 43.304 | 11.98% |
| C17 | 31.537 | 38.002 | 34.748 | 8.56% |
| C11 | 61.916 | 73.354 | 68.220 | 7.00% |
| C09 | 30.749 | 31.867 | 34.035 | 6.80% |


## 解讀檢查清單

- 區分 OOF evaluation 與重新用全部資料訓練的 production bundle。
- 不把 target runtime、oracle marker 或 future information 當作 deployment feature。
- 報告方向別結果、combined summary 與 worst cases。
- 結論只適用於目前資料、GPU 節點與 workload family。